# Find similar cases and generate clarification questions

This notebook produces two outputs for comparison:

1. **Baseline questions** based only on the new use case.
2. **Enriched questions** informed by the full text of relevant training cases.

Retrieval uses the summary prompt and embedding model shared with the indexing notebook.

In [ ]:
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from prompts import (
    SUMMARY_SYSTEM_PROMPT,
    SUMMARY_USER_TEMPLATE,
)

load_dotenv()

LLM_MODEL = os.getenv("OPENAI_LLM_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing. Copy .env.example to .env and add your key.")

client = OpenAI()

from prompts import (
    ENRICHED_QUESTION_SYSTEM_PROMPT,
    ENRICHED_QUESTION_USER_TEMPLATE,
    QUESTION_SYSTEM_PROMPT,
    QUESTION_USER_TEMPLATE,
    RELEVANCE_SYSTEM_PROMPT,
    RELEVANCE_USER_TEMPLATE,
)

## Input

Replace the example with the new use-case text.

In [ ]:
USE_CASE_TEXT = """
We want to use machine learning to improve product recommendations for customers in our online store.
""".strip()

if not USE_CASE_TEXT:
    raise ValueError("USE_CASE_TEXT must not be empty")

## Helpers

In [ ]:
def summarize_case(case_text: str) -> str:
    response = client.responses.create(
        model=LLM_MODEL,
        instructions=SUMMARY_SYSTEM_PROMPT,
        input=SUMMARY_USER_TEMPLATE.format(case_text=case_text),
    )
    return " ".join(response.output_text.strip().splitlines())


def embed_text(text: str) -> np.ndarray:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return np.asarray(response.data[0].embedding, dtype=float)


def parse_json_array(raw_text: str) -> list:
    """Parse an array while tolerating an accidental Markdown code fence."""
    text = raw_text.strip()
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        text = fenced.group(1)
    value = json.loads(text)
    if not isinstance(value, list):
        raise ValueError("The model response must be a JSON array")
    return value


def ask_for_json_array(system_prompt: str, user_prompt: str) -> list:
    response = client.responses.create(
        model=LLM_MODEL,
        instructions=system_prompt,
        input=user_prompt,
    )
    return parse_json_array(response.output_text)


def cosine_similarity(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=1)
    if query_norm == 0 or np.any(row_norms == 0):
        raise ValueError("Cannot compute cosine similarity for a zero vector")
    return (matrix @ query) / (row_norms * query_norm)

## 1. Baseline questions from the input only

In [ ]:
baseline_questions = ask_for_json_array(
    QUESTION_SYSTEM_PROMPT,
    QUESTION_USER_TEMPLATE.format(case_text=USE_CASE_TEXT),
)

print("Baseline questions")
for number, question in enumerate(baseline_questions, start=1):
    print(f"{number}. {question}")

## 2. Summarize and retrieve the five nearest cases

In [ ]:
INDEX_PATH = Path("data/train/train_index.csv")
TRAIN_DIR = Path("data/train/files")
TOP_K = int(os.getenv("CASE_FINDER_TOP_K", "5"))
if TOP_K < 1:
    raise ValueError("CASE_FINDER_TOP_K must be a positive integer")

if not INDEX_PATH.exists():
    raise FileNotFoundError("Build data/train/train_index.csv with prepare-train-data.ipynb first")

index_df = pd.read_csv(INDEX_PATH)
required_columns = {"id", "summary", "embedding"}
if not required_columns.issubset(index_df.columns):
    raise ValueError(f"Index must contain columns: {sorted(required_columns)}")
if index_df.empty:
    raise ValueError("The training index is empty")

case_summary = summarize_case(USE_CASE_TEXT)
query_embedding = embed_text(case_summary)
embedding_matrix = np.vstack([
    np.asarray(json.loads(value), dtype=float) for value in index_df["embedding"]
])
if embedding_matrix.shape[1] != query_embedding.size:
    raise ValueError(
        "Embedding dimensions differ. Rebuild the index with the configured embedding model."
    )

index_df = index_df.copy()
index_df["similarity"] = cosine_similarity(query_embedding, embedding_matrix)
top_candidates = index_df.nlargest(min(TOP_K, len(index_df)), "similarity").reset_index(drop=True)

print("Input summary:", case_summary)
top_candidates[["id", "similarity", "summary"]]

## 3. Let the LLM remove false-positive candidates

Integer positions are used in the prompt so case IDs remain opaque and cannot influence the relevance judgment.

In [ ]:
candidate_payload = "\n\n".join(
    f"Candidate {position}: {row.summary}"
    for position, row in enumerate(top_candidates.itertuples(index=False), start=1)
)
selected_positions_raw = ask_for_json_array(
    RELEVANCE_SYSTEM_PROMPT,
    RELEVANCE_USER_TEMPLATE.format(
        target_summary=case_summary,
        candidates=candidate_payload,
    ),
)

valid_positions = set(range(1, len(top_candidates) + 1))
if any(type(value) is not int or value not in valid_positions for value in selected_positions_raw):
    raise ValueError("The relevance response contains an invalid candidate ID")
selected_positions = list(dict.fromkeys(selected_positions_raw))
relevant_candidates = top_candidates.iloc[[position - 1 for position in selected_positions]].copy()

if relevant_candidates.empty:
    print("No candidate passed the LLM relevance filter.")
else:
    display(relevant_candidates[["id", "similarity", "summary"]])

## 4. Load complete reference cases and generate enriched questions

In [ ]:
case_paths_by_id = {path.stem: path for path in TRAIN_DIR.glob("*.md")}
missing_case_ids = [case_id for case_id in relevant_candidates["id"] if case_id not in case_paths_by_id]
if missing_case_ids:
    raise FileNotFoundError(f"Full case files not found for IDs: {missing_case_ids}")

reference_sections = []
for case_id in relevant_candidates["id"]:
    full_text = case_paths_by_id[case_id].read_text(encoding="utf-8")
    reference_sections.append(f"<reference id={json.dumps(case_id)}>\n{full_text}\n</reference>")

reference_cases = (
    "\n\n".join(reference_sections)
    if reference_sections
    else "No relevant reference cases were found."
)
enriched_questions = ask_for_json_array(
    ENRICHED_QUESTION_SYSTEM_PROMPT,
    ENRICHED_QUESTION_USER_TEMPLATE.format(
        case_text=USE_CASE_TEXT,
        reference_cases=reference_cases,
    ),
)

print("Enriched questions")
for number, question in enumerate(enriched_questions, start=1):
    print(f"{number}. {question}")

## Outputs for future evaluation

Use `baseline_questions` and `enriched_questions` as the two systems to compare. `top_candidates` and `relevant_candidates` preserve retrieval diagnostics for later analysis.

In [ ]:
results = {
    "input": USE_CASE_TEXT,
    "summary": case_summary,
    "baseline_questions": baseline_questions,
    "retrieved_case_ids": top_candidates["id"].tolist(),
    "relevant_case_ids": relevant_candidates["id"].tolist(),
    "enriched_questions": enriched_questions,
}
results